# 03 — Entrenamiento del Modelo

**Objetivo:** entrenar un modelo XGBoost de regresión sobre el feature store generado en el notebook 02 para predecir el retorno forward a 1 día (`Target_FwdRet_1D`) de los activos del S&P 500, evaluando su capacidad predictiva mediante una metodología robusta de validación walk-forward que evita el sesgo de look-ahead.

---

### Decisiones metodológicas

| Decisión | Justificación |
|---|---|
| **Regresión** (no clasificación) | Preserva la magnitud de la señal; permite construir carteras *long-short* por deciles; es el enfoque estándar en *cross-sectional asset pricing*. |
| **Walk-forward expanding window** | Replica las condiciones reales de producción: el modelo solo aprende del pasado. Anchored desde 2005, evaluación 2016–2024 (9 folds anuales). |
| **Purge & embargo (5 días)** | Gap entre train/val/test. Evita contaminación por solapamiento del target forward. Marcos López de Prado, *Advances in Financial ML* (2018), cap. 7. |
| **Target demeaned cross-sectional** para entrenamiento | Aísla el componente predecible (relativo entre activos) del componente macro (nivel del día). Decisión crítica documentada en la sección 3. |
| **XGBoost con `reg:pseudohubererror`** | Robusto a las colas pesadas características de los retornos diarios. Sin `eval_metric` explícita: la objective se usa también para early stopping, garantizando consistencia. |
| **Métrica principal: Rank IC (Spearman)** | Estándar en *quant equity*; robusta a outliers; mide la capacidad de ordenar correctamente los activos cada día. |
| **Optuna con objetivo IC IR** | IC IR = mean(IC) / std(IC). Prioriza consistencia sobre rentabilidad puntual, métrica que más correlaciona con Sharpe sostenible. |
| **Features: raw + ranked + macro** | XGBoost maneja redundancia vía regularización; raw y ranked aportan señales complementarias (absoluta vs. relativa al universo). Validado mediante ablación. |

### Pipeline del notebook

1. Configuración del entorno (rutas, semillas, detección de GPU).
2. Carga y verificación del feature store.
3. Definición del experimento (X, y demeaned, splits temporales).
4. Esquema walk-forward con purge & embargo.
5. Baseline lineal (Ridge) como punto de comparación honesto.
6. Tuning de hiperparámetros con Optuna (sobre el primer fold).
7. Entrenamiento walk-forward con XGBoost.
8. Evaluación: IC, IC IR, hit rate, backtest long-short por deciles.
9. Ablación: comparación con feature set rank-only.
10. Interpretabilidad: feature importance + SHAP.
11. Persistencia del modelo, predicciones y metadata.

### Entradas y salidas

| Tipo | Archivo |
|------|---------|
| Input | `data/feature_store.parquet` |
| Output | `data/predictions_oos.parquet` |
| Output | `models/xgboost_final.json` |
| Output | `data/model_training_metadata.json` |


## 1. Configuración del entorno

Inicializamos la infraestructura del notebook: imports, semillas, rutas (con resolución robusta consistente con notebooks 01/02), logging estructurado y verificación de la disponibilidad de GPU para XGBoost.

La **fijación de semillas** (`SEED = 42`) en NumPy, Optuna y XGBoost garantiza la reproducibilidad de los resultados — requisito imprescindible para que el tribunal pueda replicar las métricas reportadas en la memoria.

La **detección de CUDA** permite que el entrenamiento se ejecute sobre la GPU (NVIDIA RTX 3060 Ti), lo que reduce los tiempos de cómputo entre 5× y 10× respecto a la versión CPU sobre un dataset de 2.4M filas.


In [ ]:
import json
import logging
import pickle
import warnings
from datetime import datetime, timezone
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import xgboost as xgb
from scipy.stats import spearmanr
from sklearn.linear_model import Ridge
from sklearn.preprocessing import StandardScaler

import optuna
from optuna.samplers import TPESampler

# Silenciar logs verbosos
optuna.logging.set_verbosity(optuna.logging.WARNING)
warnings.filterwarnings("ignore", category=FutureWarning)

# --- Logging estructurado ---
logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s | %(levelname)-8s | %(message)s",
    datefmt="%H:%M:%S",
)
logger = logging.getLogger("model_training")

# --- Reproducibilidad ---
SEED = 42
np.random.seed(SEED)

In [ ]:
# --- Resolución robusta de la raíz del proyecto ---
def _find_project_root(marker: str = "data", max_levels: int = 3) -> Path:
    """Localiza la raíz del proyecto buscando el directorio 'data/'."""
    here = Path.cwd()
    for candidate in [here, *list(here.parents)[:max_levels]]:
        if (candidate / marker).is_dir():
            return candidate
    raise FileNotFoundError(
        f"No se encontró el directorio '{marker}/' partiendo de {here}."
    )

PROJECT_ROOT = _find_project_root()
DATA_DIR     = PROJECT_ROOT / "data"
MODELS_DIR   = PROJECT_ROOT / "models"
MODELS_DIR.mkdir(parents=True, exist_ok=True)

# --- Rutas ---
INPUT_FILE_PATH       = DATA_DIR / "feature_store.parquet"
PREDICTIONS_FILE_PATH = DATA_DIR / "predictions_oos.parquet"
FINAL_MODEL_PATH      = MODELS_DIR / "xgboost_final.json"
METADATA_FILE_PATH    = DATA_DIR / "model_training_metadata.json"
OPTUNA_STUDY_PATH     = MODELS_DIR / "optuna_study.pkl"

# Si existe un study antiguo (de versiones previas del notebook), lo eliminamos
# para forzar un Optuna desde cero coherente con la configuración actual.
if OPTUNA_STUDY_PATH.exists():
    OPTUNA_STUDY_PATH.unlink()
    logger.info("Optuna study previo eliminado: se ejecutará tuning desde cero.")

logger.info(f"PROJECT_ROOT : {PROJECT_ROOT}")
logger.info(f"INPUT_FILE   : {INPUT_FILE_PATH}")
logger.info(f"MODELS_DIR   : {MODELS_DIR}")

# Verificación temprana
if not INPUT_FILE_PATH.exists():
    raise FileNotFoundError(
        f"No se encontró {INPUT_FILE_PATH}. "
        "Ejecuta primero el notebook 02 para generar el feature store."
    )

In [ ]:
# --- Detección de GPU para XGBoost ---
def detect_xgb_device() -> str:
    """Devuelve 'cuda' si hay GPU disponible, 'cpu' en caso contrario."""
    try:
        test_data = xgb.DMatrix(np.array([[0.0], [1.0]]), label=np.array([0.0, 1.0]))
        xgb.train(
            {"device": "cuda", "tree_method": "hist", "verbosity": 0},
            test_data,
            num_boost_round=1,
        )
        return "cuda"
    except Exception as e:
        logger.warning(f"GPU no disponible para XGBoost ({type(e).__name__}). Usando CPU.")
        return "cpu"

XGB_DEVICE = detect_xgb_device()
logger.info(f"XGBoost device: {XGB_DEVICE}")
logger.info(f"XGBoost version: {xgb.__version__}")

## 2. Carga y verificación del feature store

Cargamos el feature store generado por el notebook 02 desde Parquet, lo que preserva el MultiIndex `(Date, Ticker)` y garantiza tipos numéricos limpios. Verificamos la integridad del dataset:

- Ausencia de `NaN` (deberían haberse eliminado en la sección 6 del notebook 02).
- Presencia del target y de las features esperadas.
- Rango temporal coherente con el de la descarga.

Cualquier desviación se considera un fallo crítico y aborta el notebook con un mensaje informativo.


In [ ]:
# Carga del feature store
df = pd.read_parquet(INPUT_FILE_PATH)
logger.info(f"Feature store cargado: {df.shape}")

# Sanity checks
assert df.isna().sum().sum() == 0, "El feature store contiene NaNs (revisar notebook 02)."
assert "Target_FwdRet_1D" in df.columns, "Falta el target Target_FwdRet_1D."
assert isinstance(df.index, pd.MultiIndex), "Se esperaba MultiIndex (Date, Ticker)."
assert df.index.names == ["Date", "Ticker"], f"Nombres de índice inesperados: {df.index.names}"

# Información descriptiva
dates_idx   = df.index.get_level_values("Date")
tickers_idx = df.index.get_level_values("Ticker")

print(f"Filas                   : {df.shape[0]:,}")
print(f"Columnas                : {df.shape[1]}")
print(f"Tickers únicos          : {tickers_idx.nunique()}")
print(f"Fechas únicas           : {dates_idx.nunique():,}")
print(f"Rango temporal          : {dates_idx.min().date()} → {dates_idx.max().date()}")
print(f"Memoria en RAM (MB)     : {df.memory_usage(deep=True).sum() / 1024**2:.2f}")
print(f"NaNs                    : {df.isna().sum().sum()}")

## 3. Definición del experimento

### 3.1 Separación de features y target

Separamos la matriz de features $X$ y el vector objetivo $y$:

$$
X \in \mathbb{R}^{N \times d}, \quad y \in \mathbb{R}^N
$$

donde $N \approx 2.4 \cdot 10^6$ es el número de pares (Fecha, Ticker) válidos y $d$ es el número de features.

**Composición de $X$:**

- 6 features **raw** (Z-Score, Amihud, Garman-Klass, RVOL, Momentum, Vol. idiosincrásica).
- 6 features **ranked** (mismas features tras `groupby(Date).rank(pct=True)`).
- Features **macro** (VIX, VIX Z-Score y sus versiones rankeadas), broadcast por fecha.

### 3.2 Decisión metodológica clave: target cross-sectionally demeaned

El target original `Target_FwdRet_1D` codifica **dos componentes** del retorno futuro:

$$
y_{t,i} = \underbrace{\bar{y}_{t}}_{\text{nivel macro del día}} + \underbrace{(y_{t,i} - \bar{y}_{t})}_{\text{componente cross-sectional}}
$$

donde $\bar{y}_t = \frac{1}{N_t}\sum_j y_{t,j}$ es el retorno medio del universo en la fecha $t$.

**Solo el segundo componente es predecible cross-sectionalmente** y es el que evalúa la métrica IC de Spearman: dada una fecha, ¿predice el modelo correctamente qué activos baten al promedio del día?

Si entrenamos XGBoost sobre el target original, el modelo descubre rápidamente que la feature macro (VIX) explica una fracción significativa de la varianza agregada (regímenes de volatilidad → retornos sesgados). Como VIX es **idéntico para todos los activos en una fecha dada**, los splits basados en VIX llevan a todos los activos del día a la misma hoja → **predicción constante por fecha** → IC indefinido (correlación de Spearman sobre vector constante = NaN).

**Solución adoptada:** entrenar sobre el target demeaned por fecha:

$$
\tilde{y}_{t,i} = y_{t,i} - \bar{y}_{t}
$$

De este modo el modelo ya no puede reducir la pérdida prediciendo el nivel del día (su media es exactamente cero por construcción) y se ve **forzado a buscar diferencias relativas entre activos**, que es el objetivo real.

> **Importante:** la **evaluación** (IC, hit rate, backtest long-short) se realiza siempre sobre el target raw `y_raw`, porque eso es lo que un trader observa en producción. El demeaning solo afecta al target de entrenamiento.

Este ajuste sigue la metodología estándar en *cross-sectional alpha modeling* (Gu, Kelly & Xiu, *Empirical Asset Pricing via Machine Learning*, 2020). Es un punto técnico fuerte para defender en la memoria, ya que demuestra el entendimiento de la diferencia entre *time-series prediction* y *cross-sectional prediction*.

### 3.3 Sanity check anti-leakage

El target ya está construido como `pct_change().shift(-1)` en el notebook 02, por lo que $y_t$ codifica el retorno de $t$ a $t+1$. Las features de la fila $t$ usan información hasta el cierre de $t$. **No hay look-ahead.**


In [ ]:
# Identificación de columnas
target_col   = "Target_FwdRet_1D"
feature_cols = [c for c in df.columns if not c.startswith("Target")]

X     = df[feature_cols].copy()
y_raw = df[target_col].copy()

# --- Target cross-sectionally demeaned (solo para entrenamiento) ---
# Aísla el componente predecible (relativo entre activos) del componente
# macro (nivel del día). Ver markdown de la sección 3.2.
y = y_raw.groupby(level="Date").transform(lambda s: s - s.mean())

# Categorización para reporting
raw_cols    = [c for c in feature_cols if not c.endswith("_rank") and not c.startswith("VIX")]
ranked_cols = [c for c in feature_cols if c.endswith("_rank") and not c.startswith("VIX")]
macro_cols  = [c for c in feature_cols if c.startswith("VIX")]

print(f"Total features          : {len(feature_cols)}")
print(f"  - Raw                 : {len(raw_cols)}")
print(f"  - Ranked              : {len(ranked_cols)}")
print(f"  - Macro (VIX)         : {len(macro_cols)}")
print(f"Target (entrenamiento)  : {target_col} (cross-sectionally demeaned)")
print(f"Target (evaluación)     : {target_col} (raw)")
print(f"X shape                 : {X.shape}")
print(f"y shape                 : {y.shape}")
print(f"\nDistribución del target demeaned (entrenamiento):")
print(y.describe().round(5))
print(f"\nDistribución del target raw (evaluación):")
print(y_raw.describe().round(5))

# Verificación: la media diaria del target demeaned debe ser ≈ 0
sample_means = y.groupby(level="Date").mean().abs()
print(f"\nMedia absoluta diaria del target demeaned (debe ser ≈ 0):")
print(f"  máx: {sample_means.max():.2e}, media: {sample_means.mean():.2e}")

## 4. Esquema de validación walk-forward

Implementamos una validación walk-forward con **expanding window** y **purge & embargo**, siguiendo la metodología propuesta por López de Prado (2018) para evitar el data leakage temporal en *time-series ML*.

### 4.1 Definición formal de los folds

Sea $T_{\min}$ el inicio del histórico (2005) y $T_{\max}$ su fin (2025). Para el fold $k$ (con $k = 1, \dots, K$):

- **Train**: $[T_{\min},\ Y_k - 1]$
- **Validation**: año $Y_k$ completo
- **Test**: año $Y_k + 1$ completo

donde $Y_k$ recorre los años de validación del 2015 al 2023, generando **9 folds**.

### 4.2 Purge & embargo

Entre los conjuntos consecutivos se introduce un gap de **5 días hábiles** (`PURGE_DAYS`). Esta separación es necesaria porque el target $y_t$ se calcula como $\text{ret}(t \to t+1)$. Sin gap, la última observación de train comparte información con la primera de validation a través del solapamiento del periodo de retorno. El purge elimina ese solapamiento y deja a los conjuntos rigurosamente independientes.

### 4.3 Ventajas frente a un único train/test split

- **Estabilidad estadística:** 9 valores de IC en lugar de uno solo, permitiendo calcular IC IR (consistencia).
- **Robustez a regímenes:** el modelo se evalúa en periodos económicamente diversos (mercado alcista 2017, COVID 2020, inflación 2022, etc.).
- **Realismo de producción:** en cada fold el modelo solo ve datos anteriores al periodo de validación/test, igual que ocurriría en operativa real.


In [ ]:
# --- Hiperparámetros del esquema walk-forward ---
TRAIN_START_YEAR   = 2005
FIRST_VAL_YEAR     = 2015
LAST_VAL_YEAR      = 2023   # fold k=9 → val=2023, test=2024
PURGE_DAYS         = 5      # gap entre train/val/test (López de Prado, AFML cap. 7)


def generate_walk_forward_folds(
    dates: pd.DatetimeIndex,
    train_start: int,
    first_val_year: int,
    last_val_year: int,
    purge_days: int,
) -> list[dict]:
    """
    Genera los folds de validación walk-forward con expanding window.

    Parameters
    ----------
    dates : pd.DatetimeIndex
        Índice temporal único y ordenado del dataset.
    train_start : int
        Año de inicio del primer train (anchored).
    first_val_year, last_val_year : int
        Rango de años usados como validación (inclusivo).
    purge_days : int
        Días hábiles de gap entre train/val/test.

    Returns
    -------
    list[dict]
        Cada fold es un dict con claves:
        {fold, train_start, train_end, val_start, val_end, test_start, test_end}.
    """
    folds = []
    sorted_dates = pd.DatetimeIndex(sorted(set(dates)))

    for i, val_year in enumerate(range(first_val_year, last_val_year + 1), start=1):
        val_start_target  = pd.Timestamp(f"{val_year}-01-01")
        val_end_target    = pd.Timestamp(f"{val_year}-12-31")
        test_start_target = pd.Timestamp(f"{val_year + 1}-01-01")
        test_end_target   = pd.Timestamp(f"{val_year + 1}-12-31")

        # Snap a fechas reales presentes en el dataset
        val_dates  = sorted_dates[(sorted_dates >= val_start_target)  & (sorted_dates <= val_end_target)]
        test_dates = sorted_dates[(sorted_dates >= test_start_target) & (sorted_dates <= test_end_target)]
        if len(val_dates) == 0 or len(test_dates) == 0:
            continue
        val_start, val_end   = val_dates[0],  val_dates[-1]
        test_start, test_end = test_dates[0], test_dates[-1]

        # Train: desde train_start hasta (val_start - purge_days hábiles)
        train_end_idx = sorted_dates.get_indexer([val_start])[0] - purge_days - 1
        if train_end_idx <= 0:
            continue
        train_end = sorted_dates[train_end_idx]

        # Embargo entre val y test
        test_start_idx     = sorted_dates.get_indexer([test_start])[0]
        val_end_purged_idx = test_start_idx - purge_days - 1
        if val_end_purged_idx <= 0:
            continue
        val_end_purged = sorted_dates[val_end_purged_idx]
        # Si el purge "se come" toda la val, usar la val original (caso borde)
        if val_end_purged < val_start:
            val_end_purged = val_end

        folds.append({
            "fold":        i,
            "train_start": pd.Timestamp(f"{train_start}-01-01"),
            "train_end":   train_end,
            "val_start":   val_start,
            "val_end":     val_end_purged,
            "test_start":  test_start,
            "test_end":    test_end,
        })

    return folds


# Generación
unique_dates = pd.DatetimeIndex(sorted(set(dates_idx)))
FOLDS = generate_walk_forward_folds(
    dates=unique_dates,
    train_start=TRAIN_START_YEAR,
    first_val_year=FIRST_VAL_YEAR,
    last_val_year=LAST_VAL_YEAR,
    purge_days=PURGE_DAYS,
)
logger.info(f"Folds generados: {len(FOLDS)}")

# Tabla resumen
folds_df = pd.DataFrame(FOLDS)
folds_df_show = folds_df.copy()
for col in ["train_start", "train_end", "val_start", "val_end", "test_start", "test_end"]:
    folds_df_show[col] = folds_df_show[col].dt.date
print(folds_df_show.to_string(index=False))

### 4.4 Visualización del esquema

Visualización gráfica de los 9 folds. Cada barra representa un fold y muestra los tres periodos (Train en azul, Val en naranja, Test en verde) y el gap de purge entre ellos. Esta figura puede incluirse directamente en la memoria del TFG.


In [ ]:
def plot_walk_forward_scheme(folds: list[dict]) -> None:
    """Visualiza los folds walk-forward en un diagrama horizontal."""
    fig, ax = plt.subplots(figsize=(13, 0.6 * len(folds) + 1))
    colors = {"train": "#2E86AB", "val": "#F18F01", "test": "#43AA8B"}

    for f in folds:
        y_pos = f["fold"]
        ax.barh(y_pos, (f["train_end"] - f["train_start"]).days,
                left=f["train_start"], color=colors["train"], height=0.6, label="Train")
        ax.barh(y_pos, (f["val_end"] - f["val_start"]).days,
                left=f["val_start"], color=colors["val"], height=0.6, label="Val")
        ax.barh(y_pos, (f["test_end"] - f["test_start"]).days,
                left=f["test_start"], color=colors["test"], height=0.6, label="Test")

    handles, labels = ax.get_legend_handles_labels()
    by_label = dict(zip(labels, handles))
    ax.legend(by_label.values(), by_label.keys(), loc="upper left", framealpha=0.95)

    ax.set_yticks([f["fold"] for f in folds])
    ax.set_yticklabels([f"Fold {f['fold']}" for f in folds])
    ax.invert_yaxis()
    ax.set_xlabel("Fecha")
    ax.set_title(f"Esquema de validación walk-forward (expanding window, purge = {PURGE_DAYS} días)")
    ax.grid(axis="x", alpha=0.3)
    plt.tight_layout()
    plt.show()


plot_walk_forward_scheme(FOLDS)

## 5. Baseline lineal (Ridge) y métricas de evaluación

### 5.1 Métricas

Definimos el conjunto de métricas que usaremos a lo largo del notebook. **Todas se calculan sobre el target raw `y_raw`**, que es el que un trader observa en producción.

#### Information Coefficient (IC)
$$
\text{IC}_t = \rho_S\bigl(\hat{y}_{t,\cdot},\ y_{t,\cdot}\bigr)
$$

Correlación de **Spearman** entre predicciones y realizaciones, computada *cross-sectional* en cada fecha $t$ sobre todos los activos disponibles. Métrica estándar en *cross-sectional asset pricing* y robusta a outliers.

#### IC IR (Information Ratio del IC)
$$
\text{IC IR} = \frac{\overline{\text{IC}}}{\sigma(\text{IC})}
$$

Mide la **consistencia** del IC a lo largo del tiempo. Un IC IR > 0.5 anualizado se considera bueno; > 1.0 es excelente.

#### Hit rate direccional
$$
\text{Hit Rate} = \frac{1}{N}\sum_t \mathbb{1}\!\left[\,\text{sgn}(\hat{y}_t) = \text{sgn}(y_t)\,\right]
$$

Proporción de aciertos en el signo. Útil como métrica complementaria interpretable.

### 5.2 Baseline lineal (Ridge)

Antes de entrenar XGBoost, computamos un **baseline honesto** con regresión Ridge sobre las mismas features y los mismos folds, **entrenado también con el target demeaned** para comparabilidad. Esto nos permite responder a una pregunta crítica: **¿el feature engineering captura señal predictiva, o el modelo XGBoost simplemente está sobre-ajustando ruido?**

Si XGBoost no bate de forma consistente al baseline lineal, hay que replantear el feature engineering antes que el modelo. Es el control de calidad metodológico básico que cualquier tribunal técnico va a exigir.

Se aplica `StandardScaler` antes de Ridge porque la regresión lineal regularizada es sensible a la escala de las features (cosa que XGBoost no es).


In [ ]:
# --- Funciones de métricas ---
def compute_ic_per_date(y_true: pd.Series, y_pred: pd.Series) -> pd.Series:
    """
    Calcula el Information Coefficient (Spearman) por fecha.

    Parameters
    ----------
    y_true, y_pred : pd.Series con MultiIndex (Date, Ticker)

    Returns
    -------
    pd.Series indexada por fecha con el IC de cada día.
    """
    df_ic = pd.DataFrame({"y_true": y_true, "y_pred": y_pred})
    ic_per_date = df_ic.groupby(level="Date").apply(
        lambda g: spearmanr(g["y_true"], g["y_pred"]).correlation
        if len(g) >= 5 and g["y_pred"].nunique() > 1 and g["y_true"].nunique() > 1
        else np.nan
    )
    return ic_per_date.dropna()


def compute_metrics(y_true: pd.Series, y_pred: pd.Series) -> dict:
    """Calcula el conjunto completo de métricas de evaluación."""
    ic_series = compute_ic_per_date(y_true, y_pred)
    hit_rate  = (np.sign(y_true.values) == np.sign(y_pred.values)).mean()
    return {
        "ic_mean":    float(ic_series.mean()),
        "ic_std":     float(ic_series.std()),
        "ic_ir":      float(ic_series.mean() / ic_series.std()) if ic_series.std() > 0 else 0.0,
        "ic_t_stat":  float(ic_series.mean() / (ic_series.std() / np.sqrt(len(ic_series)))) if ic_series.std() > 0 else 0.0,
        "hit_rate":   float(hit_rate),
        "n_dates":    int(len(ic_series)),
    }

In [ ]:
# --- Baseline Ridge sobre walk-forward ---
def slice_fold(
    X: pd.DataFrame, y: pd.Series, y_raw: pd.Series, fold: dict
) -> dict:
    """
    Extrae los conjuntos train/val/test para un fold dado.

    Devuelve tanto el target demeaned `y` (para entrenamiento) como el target
    raw `y_raw` (para evaluación). Ver sección 3.2.
    """
    dates = X.index.get_level_values("Date")
    train_mask = (dates >= fold["train_start"]) & (dates <= fold["train_end"])
    val_mask   = (dates >= fold["val_start"])   & (dates <= fold["val_end"])
    test_mask  = (dates >= fold["test_start"])  & (dates <= fold["test_end"])
    return {
        "X_train":    X[train_mask],     "y_train":    y[train_mask],
        "X_val":      X[val_mask],       "y_val":      y[val_mask],
        "X_test":     X[test_mask],      "y_test":     y[test_mask],
        "y_val_raw":  y_raw[val_mask],
        "y_test_raw": y_raw[test_mask],
    }


def run_ridge_baseline(
    X: pd.DataFrame, y: pd.Series, y_raw: pd.Series, folds: list[dict]
) -> pd.DataFrame:
    """Entrena un Ridge en cada fold y devuelve las predicciones OOS concatenadas."""
    preds_list = []
    for fold in folds:
        s = slice_fold(X, y, y_raw, fold)

        # Escalado fit en train, transform en test (sin leakage)
        scaler = StandardScaler().fit(s["X_train"])
        X_tr = scaler.transform(s["X_train"])
        X_te = scaler.transform(s["X_test"])

        # Entrena con y demeaned
        model = Ridge(alpha=1.0, random_state=SEED)
        model.fit(X_tr, s["y_train"])
        preds_test = model.predict(X_te)

        # Evalúa con y_raw
        df_preds = pd.DataFrame({
            "y_true":   s["y_test_raw"].values,
            "y_pred":   preds_test,
            "fold":     fold["fold"],
        }, index=s["y_test_raw"].index)
        preds_list.append(df_preds)
        logger.info(f"Ridge fold {fold['fold']}: train={len(s['y_train']):,} test={len(s['y_test']):,}")

    return pd.concat(preds_list)


logger.info("Entrenando baseline Ridge en los 9 folds...")
ridge_preds = run_ridge_baseline(X, y, y_raw, FOLDS)
ridge_metrics = compute_metrics(ridge_preds["y_true"], ridge_preds["y_pred"])

print("\n--- Baseline Ridge (out-of-sample 2016–2024) ---")
for k, v in ridge_metrics.items():
    print(f"  {k:15s}: {v:.5f}" if isinstance(v, float) else f"  {k:15s}: {v}")

## 6. Tuning de hiperparámetros con Optuna

### 6.1 Estrategia

Optimizamos los hiperparámetros de XGBoost usando **Optuna** (sampler TPE, *Tree-structured Parzen Estimator*) sobre el **primer fold** del esquema walk-forward (train: 2005–2014, validación: 2015). Los mejores hiperparámetros encontrados se aplican fijos a los 9 folds restantes.

**¿Por qué solo en el primer fold?** Re-tunear en cada fold sería puristamente correcto pero multiplicaría por 9 el tiempo de cómputo y aumentaría el riesgo de sobreajuste por configuraciones de Optuna. Tunear en el fold más antiguo es la práctica estándar en la literatura aplicada (Gu, Kelly & Xiu 2020) y mantiene la honestidad metodológica: las decisiones de hiperparámetros se toman antes de ver los datos out-of-sample.

### 6.2 Función objetivo

Optuna **maximiza el IC IR** sobre el conjunto de validación del fold 1 (calculado contra `y_val_raw`). IC IR penaliza la inestabilidad del IC: una configuración con IC = 0.05 estable día a día es preferible a otra con IC = 0.08 pero alta varianza. Esta es la métrica que mejor predice el Sharpe del backtest final.

### 6.3 Configuración de XGBoost

- **`objective = reg:pseudohubererror`**: robusto a las colas pesadas de los retornos diarios.
- **Sin `eval_metric` explícita**: XGBoost usa la objective como métrica de validación interna, garantizando consistencia entre lo que se optimiza durante el entrenamiento y lo que dispara el early stopping.
- **`tree_method = hist`** + **`device = cuda`**: aprovecha la GPU para acelerar el entrenamiento sobre 1M+ filas.

### 6.4 Search space

| Parámetro | Rango | Justificación |
|---|---|---|
| `max_depth` | 3–8 | Trees profundos sobre-ajustan en datos financieros ruidosos. |
| `learning_rate` | 0.01–0.2 (log) | Compromiso entre velocidad y estabilidad. |
| `subsample` | 0.6–1.0 | Bagging sobre filas, regularización implícita. |
| `colsample_bytree` | 0.5–1.0 | Bagging sobre features. |
| `min_child_weight` | 1–10 | Evita splits sobre muestras minúsculas (ruido). Rango recortado para evitar la zona "degenerada" donde ningún split supera la regularización. |
| `reg_alpha` (L1) | 1e-3–10 (log) | Sparsity — útil si features redundantes. |
| `reg_lambda` (L2) | 1e-3–10 (log) | Regularización suave. |
| `gamma` | 0–0.5 | Mínima ganancia para splittear. Acotado para evitar árboles colapsados a un único nodo. |

`n_estimators` se gestiona dinámicamente con **early stopping** (patience = 100): el booster crece hasta que la métrica de validación deja de mejorar.


In [ ]:
# --- Configuración de Optuna ---
N_OPTUNA_TRIALS    = 50           # ~25-40 min en RTX 3060 Ti
EARLY_STOP_ROUNDS  = 100
MAX_BOOST_ROUNDS   = 1500

# Habilitar logs informativos de Optuna durante el tuning
optuna.logging.set_verbosity(optuna.logging.INFO)


def xgb_base_params(device: str = XGB_DEVICE) -> dict:
    """Hiperparámetros base de XGBoost compartidos en todos los entrenamientos.

    Notas:
    - `eval_metric` se omite intencionalmente: XGBoost usará la objective
      Pseudo-Huber como métrica de validación interna, garantizando consistencia
      entre lo que se optimiza durante el entrenamiento y lo que dispara el
      early stopping.
    """
    return {
        "objective":    "reg:pseudohubererror",  # robusto a colas pesadas
        "huber_slope":  1.0,                      # delta de Huber
        "device":       device,
        "tree_method":  "hist",
        "verbosity":    0,
        "seed":         SEED,
    }


def optuna_objective(trial: optuna.Trial, fold: dict) -> float:
    """Función objetivo: maximizar IC IR en el set de validación del fold."""
    s = slice_fold(X, y, y_raw, fold)

    params = {
        **xgb_base_params(),
        "max_depth":        trial.suggest_int("max_depth", 3, 8),
        "learning_rate":    trial.suggest_float("learning_rate", 0.01, 0.2, log=True),
        "subsample":        trial.suggest_float("subsample", 0.6, 1.0),
        "colsample_bytree": trial.suggest_float("colsample_bytree", 0.5, 1.0),
        "min_child_weight": trial.suggest_int("min_child_weight", 1, 10),
        "reg_alpha":        trial.suggest_float("reg_alpha", 1e-3, 10.0, log=True),
        "reg_lambda":       trial.suggest_float("reg_lambda", 1e-3, 10.0, log=True),
        "gamma":            trial.suggest_float("gamma", 0.0, 0.5),
    }

    dtrain = xgb.DMatrix(s["X_train"], label=s["y_train"])
    dval   = xgb.DMatrix(s["X_val"],   label=s["y_val"])

    booster = xgb.train(
        params,
        dtrain,
        num_boost_round=MAX_BOOST_ROUNDS,
        evals=[(dval, "val")],
        early_stopping_rounds=EARLY_STOP_ROUNDS,
        verbose_eval=False,
    )

    # IC se calcula contra y_raw (target real, no demeaned)
    val_pred  = pd.Series(booster.predict(dval), index=s["y_val_raw"].index)
    ic_series = compute_ic_per_date(s["y_val_raw"], val_pred)
    if len(ic_series) < 30 or ic_series.std() == 0:
        return -10.0
    return float(ic_series.mean() / ic_series.std())


# Estudio Optuna sobre el primer fold
fold_for_tuning = FOLDS[0]
logger.info(
    f"Optuna sobre fold 1: train={fold_for_tuning['train_start'].date()}→{fold_for_tuning['train_end'].date()}, "
    f"val={fold_for_tuning['val_start'].date()}→{fold_for_tuning['val_end'].date()}"
)

study = optuna.create_study(
    direction="maximize",
    sampler=TPESampler(seed=SEED),
    study_name="xgboost_walkforward_tfg",
)
study.optimize(
    lambda trial: optuna_objective(trial, fold_for_tuning),
    n_trials=N_OPTUNA_TRIALS,
    show_progress_bar=True,
)

# Persistir el study
with open(OPTUNA_STUDY_PATH, "wb") as f:
    pickle.dump(study, f)
logger.info(f"Optuna study guardado en: {OPTUNA_STUDY_PATH}")

# Resultados
BEST_PARAMS = {**xgb_base_params(), **study.best_params}
print(f"\nMejor IC IR en validación: {study.best_value:.4f}")
print("Hiperparámetros óptimos:")
for k, v in study.best_params.items():
    print(f"  {k:20s}: {v}")

# Volver a silenciar Optuna
optuna.logging.set_verbosity(optuna.logging.WARNING)

### 6.5 Análisis del proceso de optimización

Visualizamos la evolución del IC IR a lo largo de los trials de Optuna y la importancia relativa de cada hiperparámetro. Estas dos figuras documentan que la búsqueda fue informativa (no plana) y que los hiperparámetros más sensibles guían la decisión de fijar la configuración.


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 4.5))

# --- (a) Historia de optimización ---
trial_values = [t.value for t in study.trials if t.value is not None]
running_max  = np.maximum.accumulate(trial_values)
axes[0].plot(trial_values, "o", alpha=0.4, label="IC IR por trial")
axes[0].plot(running_max, "-", color="darkred", lw=2, label="Mejor acumulado")
axes[0].set_xlabel("Trial")
axes[0].set_ylabel("IC IR (validación fold 1)")
axes[0].set_title("Evolución de la optimización con Optuna")
axes[0].legend()
axes[0].grid(alpha=0.3)

# --- (b) Importancia de hiperparámetros ---
try:
    importances = optuna.importance.get_param_importances(study)
    params_imp = list(importances.keys())
    values_imp = list(importances.values())
    axes[1].barh(params_imp, values_imp, color="#2E86AB")
    axes[1].invert_yaxis()
    axes[1].set_xlabel("Importancia")
    axes[1].set_title("Importancia de hiperparámetros")
    axes[1].grid(axis="x", alpha=0.3)
except Exception as e:
    axes[1].text(0.5, 0.5, f"Importancia no disponible:\n{e}",
                 ha="center", va="center", transform=axes[1].transAxes)

plt.tight_layout()
plt.show()

## 7. Entrenamiento walk-forward con XGBoost

Aplicamos los hiperparámetros óptimos hallados por Optuna a los 9 folds del esquema walk-forward. Para cada fold:

1. Construimos `DMatrix` para train, val y test (usando GPU si está disponible).
2. Entrenamos XGBoost con **early stopping** sobre el set de validación (patience = 100).
3. Predecimos sobre el set de test (datos completamente unseen).
4. Almacenamos las predicciones junto con las realizaciones del target raw.

Las predicciones de los 9 folds se concatenan formando un **panel out-of-sample** completo de 2016 a 2024 (~9 años de evaluación robusta).

> Recordatorio metodológico: el modelo entrena con el target demeaned `y` pero las predicciones se evalúan contra `y_raw`. Esto significa que **las predicciones del modelo aproximan retornos relativos al promedio del día**, lo que es exactamente lo que necesita el backtest long-short por deciles.


In [ ]:
def train_xgboost_walk_forward(
    X: pd.DataFrame,
    y: pd.Series,
    y_raw: pd.Series,
    folds: list[dict],
    params: dict,
    label: str = "XGBoost",
) -> tuple[pd.DataFrame, list[xgb.Booster]]:
    """
    Entrena XGBoost en cada fold y concatena predicciones out-of-sample.

    El entrenamiento usa el target demeaned `y`; la evaluación posterior se
    realiza contra el target raw `y_raw`.

    Returns
    -------
    (predictions_df, models_list)
    """
    preds_list = []
    models = []
    for fold in folds:
        s = slice_fold(X, y, y_raw, fold)

        dtrain = xgb.DMatrix(s["X_train"], label=s["y_train"])
        dval   = xgb.DMatrix(s["X_val"],   label=s["y_val"])
        dtest  = xgb.DMatrix(s["X_test"],  label=s["y_test"])

        booster = xgb.train(
            params,
            dtrain,
            num_boost_round=MAX_BOOST_ROUNDS,
            evals=[(dval, "val")],
            early_stopping_rounds=EARLY_STOP_ROUNDS,
            verbose_eval=False,
        )
        models.append(booster)

        preds_test = booster.predict(dtest)

        df_preds = pd.DataFrame({
            "y_true": s["y_test_raw"].values,   # evaluación contra raw
            "y_pred": preds_test,
            "fold":   fold["fold"],
        }, index=s["y_test_raw"].index)
        preds_list.append(df_preds)

        logger.info(
            f"{label} fold {fold['fold']}: "
            f"train={len(s['y_train']):,} val={len(s['y_val']):,} test={len(s['y_test']):,} | "
            f"best_iter={booster.best_iteration}"
        )

    return pd.concat(preds_list), models


logger.info("Entrenando XGBoost walk-forward (9 folds)...")
xgb_preds, xgb_models = train_xgboost_walk_forward(X, y, y_raw, FOLDS, BEST_PARAMS, label="XGBoost")
xgb_metrics = compute_metrics(xgb_preds["y_true"], xgb_preds["y_pred"])

print("\n--- XGBoost walk-forward (out-of-sample 2016–2024) ---")
for k, v in xgb_metrics.items():
    print(f"  {k:15s}: {v:.5f}" if isinstance(v, float) else f"  {k:15s}: {v}")

## 8. Evaluación

### 8.1 Comparación XGBoost vs. baseline Ridge

Comparación directa de las métricas globales out-of-sample.


In [ ]:
metrics_df = pd.DataFrame({
    "Ridge":   ridge_metrics,
    "XGBoost": xgb_metrics,
}).T
metrics_df = metrics_df[["ic_mean", "ic_std", "ic_ir", "ic_t_stat", "hit_rate", "n_dates"]]

# Mejora porcentual del IC IR
ic_ir_improvement = (
    (xgb_metrics["ic_ir"] - ridge_metrics["ic_ir"]) / abs(ridge_metrics["ic_ir"]) * 100
    if ridge_metrics["ic_ir"] != 0 else 0.0
)

print("--- Tabla comparativa ---")
print(metrics_df.round(5).to_string())
print(f"\nMejora del IC IR XGBoost vs Ridge: {ic_ir_improvement:+.1f}%")

### 8.2 Distribución del IC diario

Histograma del IC por fecha y serie temporal acumulada. Permiten ver visualmente la consistencia de la señal a lo largo del tiempo.


In [ ]:
ic_xgb   = compute_ic_per_date(xgb_preds["y_true"],   xgb_preds["y_pred"])
ic_ridge = compute_ic_per_date(ridge_preds["y_true"], ridge_preds["y_pred"])

fig, axes = plt.subplots(1, 2, figsize=(14, 4.5))

# (a) Histograma comparativo
axes[0].hist(ic_ridge, bins=40, alpha=0.55, label=f"Ridge  (μ={ic_ridge.mean():.4f})",
             color="#888888", edgecolor="white")
axes[0].hist(ic_xgb,   bins=40, alpha=0.65, label=f"XGBoost (μ={ic_xgb.mean():.4f})",
             color="#2E86AB", edgecolor="white")
axes[0].axvline(0,                color="black",   ls="--", lw=1, alpha=0.6)
axes[0].axvline(ic_xgb.mean(),    color="#2E86AB", ls="--", lw=2)
axes[0].axvline(ic_ridge.mean(),  color="#888888", ls="--", lw=2)
axes[0].set_xlabel("IC diario (Spearman)")
axes[0].set_ylabel("Frecuencia")
axes[0].set_title("Distribución del IC diario (out-of-sample)")
axes[0].legend()
axes[0].grid(alpha=0.3)

# (b) IC acumulado
axes[1].plot(ic_xgb.cumsum().index,   ic_xgb.cumsum().values,
             label="XGBoost", color="#2E86AB", lw=1.8)
axes[1].plot(ic_ridge.cumsum().index, ic_ridge.cumsum().values,
             label="Ridge",   color="#888888", lw=1.5)
axes[1].set_xlabel("Fecha")
axes[1].set_ylabel("IC acumulado")
axes[1].set_title("IC acumulado en el tiempo")
axes[1].legend()
axes[1].grid(alpha=0.3)

plt.tight_layout()
plt.show()

### 8.3 Estabilidad por año

Métricas calculadas por año natural sobre el período 2016–2024. Permite identificar si el modelo es consistente o si su rendimiento depende fuertemente del régimen de mercado.


In [ ]:
def metrics_by_year(preds: pd.DataFrame) -> pd.DataFrame:
    """Calcula métricas separadas por año natural."""
    rows = []
    for year, group in preds.groupby(preds.index.get_level_values("Date").year):
        m = compute_metrics(group["y_true"], group["y_pred"])
        m["year"] = year
        rows.append(m)
    return pd.DataFrame(rows).set_index("year")[["ic_mean", "ic_ir", "hit_rate", "n_dates"]]


yearly_xgb = metrics_by_year(xgb_preds)
print("--- XGBoost: métricas por año ---")
print(yearly_xgb.round(4).to_string())

# Visualización
fig, ax = plt.subplots(figsize=(11, 4.5))
yearly_xgb["ic_mean"].plot(kind="bar", ax=ax, color="#2E86AB", edgecolor="white", alpha=0.85)
ax.axhline(0, color="black", lw=1)
ax.axhline(yearly_xgb["ic_mean"].mean(), color="darkred", ls="--",
           label=f"Media global: {yearly_xgb['ic_mean'].mean():.4f}")
ax.set_xlabel("Año")
ax.set_ylabel("IC medio")
ax.set_title("Estabilidad anual del IC (XGBoost, out-of-sample)")
ax.legend()
ax.grid(axis="y", alpha=0.3)
plt.xticks(rotation=0)
plt.tight_layout()
plt.show()

### 8.4 Backtest long-short por deciles

Evaluamos la utilidad económica de las predicciones construyendo una **cartera long-short cross-sectional** rebalanceada diariamente:

1. Cada día, ordenamos los activos por la predicción del modelo.
2. **Long**: top decil (10% con predicción más alta).
3. **Short**: bottom decil (10% con predicción más baja).
4. Retorno de la cartera = retorno medio del long − retorno medio del short.
5. Asumimos pesos iguales dentro de cada cartera y costes de transacción nulos (asunción simplificadora documentada como limitación).

Esta evaluación va más allá del IC: convierte la señal estadística en métricas financieras directamente comparables con benchmarks de la industria (Sharpe, max drawdown).


In [ ]:
def long_short_backtest(preds: pd.DataFrame, n_quantiles: int = 10) -> pd.Series:
    """
    Backtest long-short por deciles.

    Returns
    -------
    pd.Series con el retorno diario del portafolio long-short.
    """
    df_bt = preds.copy()
    df_bt["quantile"] = (
        df_bt.groupby(level="Date")["y_pred"]
             .transform(lambda x: pd.qcut(x, n_quantiles, labels=False, duplicates="drop"))
    )
    longs  = df_bt[df_bt["quantile"] == n_quantiles - 1].groupby(level="Date")["y_true"].mean()
    shorts = df_bt[df_bt["quantile"] == 0                ].groupby(level="Date")["y_true"].mean()
    return (longs - shorts).rename("ls_return")


def portfolio_metrics(returns: pd.Series, periods_per_year: int = 252) -> dict:
    """Métricas estándar de portfolio: retorno anual, Sharpe, max drawdown."""
    ann_return = returns.mean() * periods_per_year
    ann_vol    = returns.std()  * np.sqrt(periods_per_year)
    sharpe     = ann_return / ann_vol if ann_vol > 0 else 0.0
    cumulative = (1 + returns).cumprod()
    drawdown   = (cumulative / cumulative.cummax() - 1).min()
    return {
        "annualized_return":   float(ann_return),
        "annualized_vol":      float(ann_vol),
        "sharpe_ratio":        float(sharpe),
        "max_drawdown":        float(drawdown),
        "cumulative_return":   float(cumulative.iloc[-1] - 1),
    }


ls_xgb   = long_short_backtest(xgb_preds)
ls_ridge = long_short_backtest(ridge_preds)

m_xgb   = portfolio_metrics(ls_xgb)
m_ridge = portfolio_metrics(ls_ridge)

bt_df = pd.DataFrame({"Ridge": m_ridge, "XGBoost": m_xgb}).T
print("--- Backtest long-short por deciles (sin costes de transacción) ---")
print(bt_df.round(4).to_string())

# Equity curves
fig, ax = plt.subplots(figsize=(12, 5))
(1 + ls_xgb).cumprod().plot(ax=ax,   label=f"XGBoost (Sharpe={m_xgb['sharpe_ratio']:.2f})", color="#2E86AB", lw=1.8)
(1 + ls_ridge).cumprod().plot(ax=ax, label=f"Ridge   (Sharpe={m_ridge['sharpe_ratio']:.2f})", color="#888888", lw=1.5)
ax.axhline(1, color="black", ls="--", lw=1, alpha=0.5)
ax.set_xlabel("Fecha")
ax.set_ylabel("Equity (base 1.0)")
ax.set_title("Curva de capital — cartera long-short por deciles")
ax.legend()
ax.grid(alpha=0.3)
plt.tight_layout()
plt.show()

## 9. Ablación: rank-only vs. modelo completo

Validamos empíricamente la decisión de incluir features raw + ranked en lugar de solo ranked. Re-entrenamos un modelo XGBoost idéntico (mismos hiperparámetros y mismos folds) restringiendo el feature set a **solo features ranked + macro**.

Si el modelo completo bate al rank-only en IC IR sobre el conjunto de test, queda demostrado que las features raw aportan señal complementaria. En caso contrario, el rank-only sería la elección preferible (Occam's razor: menos features, menos riesgo de overfitting).


In [ ]:
# Subset de features rank-only (incluye macro)
rank_only_cols = ranked_cols + macro_cols
X_rank = X[rank_only_cols].copy()

logger.info(f"Ablación rank-only: {len(rank_only_cols)} features → {rank_only_cols}")
logger.info("Entrenando XGBoost rank-only walk-forward (9 folds)...")

xgb_preds_rank, _ = train_xgboost_walk_forward(
    X_rank, y, y_raw, FOLDS, BEST_PARAMS, label="XGBoost-RankOnly"
)
metrics_rank = compute_metrics(xgb_preds_rank["y_true"], xgb_preds_rank["y_pred"])
ls_rank = long_short_backtest(xgb_preds_rank)
m_rank = portfolio_metrics(ls_rank)

# Tabla comparativa final
ablation_df = pd.DataFrame({
    "Ridge (full)":           {**ridge_metrics, **m_ridge},
    "XGBoost (rank-only)":    {**metrics_rank,  **m_rank},
    "XGBoost (full)":         {**xgb_metrics,   **m_xgb},
}).T
cols_show = ["ic_mean", "ic_ir", "hit_rate", "sharpe_ratio", "max_drawdown", "annualized_return"]
print("--- Ablación: comparación de feature sets ---")
print(ablation_df[cols_show].round(4).to_string())

## 10. Interpretabilidad

Un modelo predictivo, por bueno que sea su rendimiento, tiene poco valor si funciona como caja negra. Aplicamos dos técnicas complementarias:

- **Feature importance (gain)**: agregada a través de los 9 folds. Indica cuánto contribuye cada feature a reducir el error en los splits del árbol.
- **IC marginal por feature**: correlación de Spearman entre cada feature individual y el target en los datos de test. Identifica las features con señal predictiva univariada.

Una feature útil debería destacar en al menos uno de los dos análisis. Las features con alta importancia pero IC marginal nulo capturan **interacciones no lineales** — uno de los argumentos a favor de XGBoost frente a un modelo lineal.


In [ ]:
# --- Feature importance agregada a través de folds ---
def aggregate_feature_importance(models: list[xgb.Booster], features: list[str]) -> pd.DataFrame:
    """Agrega la importancia 'gain' a través de todos los modelos del walk-forward."""
    accum = {f: 0.0 for f in features}
    for booster in models:
        imp = booster.get_score(importance_type="gain")
        for f, v in imp.items():
            if f in accum:
                accum[f] += v
    importance_df = (
        pd.DataFrame(list(accum.items()), columns=["feature", "gain"])
          .sort_values("gain", ascending=False)
          .reset_index(drop=True)
    )
    total = importance_df["gain"].sum()
    importance_df["gain_pct"] = importance_df["gain"] / total * 100 if total > 0 else 0
    return importance_df


fi_df = aggregate_feature_importance(xgb_models, feature_cols)

# --- IC marginal por feature (en el conjunto de test concatenado) ---
test_dates_mask = X.index.get_level_values("Date") >= FOLDS[0]["test_start"]
X_test_full = X[test_dates_mask]
y_test_full = y_raw[test_dates_mask]

marginal_ic = {
    f: compute_ic_per_date(y_test_full, X_test_full[f]).mean()
    for f in feature_cols
}
mic_df = (
    pd.DataFrame(list(marginal_ic.items()), columns=["feature", "marginal_ic"])
      .sort_values("marginal_ic", key=abs, ascending=False)
      .reset_index(drop=True)
)

# Visualización combinada
fig, axes = plt.subplots(1, 2, figsize=(14, 5.5))

axes[0].barh(fi_df["feature"], fi_df["gain_pct"], color="#2E86AB", edgecolor="white")
axes[0].invert_yaxis()
axes[0].set_xlabel("Gain agregado (%)")
axes[0].set_title("Importancia de features (XGBoost, agregada en folds)")
axes[0].grid(axis="x", alpha=0.3)

colors = ["#43AA8B" if v > 0 else "#E63946" for v in mic_df["marginal_ic"]]
axes[1].barh(mic_df["feature"], mic_df["marginal_ic"], color=colors, edgecolor="white")
axes[1].invert_yaxis()
axes[1].axvline(0, color="black", lw=1)
axes[1].set_xlabel("IC marginal medio")
axes[1].set_title("IC marginal por feature (test 2016–2024)")
axes[1].grid(axis="x", alpha=0.3)

plt.tight_layout()
plt.show()

print("\n--- Top features por gain ---")
print(fi_df.round(3).to_string(index=False))

### 10.1 Análisis SHAP (opcional)

SHAP (*SHapley Additive exPlanations*) descompone cada predicción individual en la contribución de cada feature. Calculamos los valores SHAP sobre una **muestra estratificada por fecha** del test set para mantener tiempos de cómputo razonables.

> Si SHAP no está instalado o el cálculo excede los recursos disponibles, esta sección puede omitirse — la importancia agregada de la sección anterior ya proporciona interpretabilidad suficiente.


In [ ]:
# Análisis SHAP sobre muestra del test
try:
    import shap

    # Muestra estratificada: ~200 obs por mes del test
    test_idx = X.index.get_level_values("Date") >= FOLDS[0]["test_start"]
    X_test_all = X[test_idx]
    sample = (
        X_test_all.groupby(X_test_all.index.get_level_values("Date").to_period("M"), group_keys=False)
                  .apply(lambda g: g.sample(min(len(g), 200), random_state=SEED))
    )
    logger.info(f"Calculando SHAP sobre muestra de {len(sample):,} observaciones...")

    # Usamos el modelo del último fold (entrenado con el histórico más amplio)
    last_model = xgb_models[-1]
    explainer  = shap.TreeExplainer(last_model)
    shap_values = explainer.shap_values(sample)

    # Summary plot
    plt.figure(figsize=(10, 6))
    shap.summary_plot(shap_values, sample, plot_type="bar", show=False)
    plt.title("SHAP — importancia global (último modelo, muestra del test)")
    plt.tight_layout()
    plt.show()

    plt.figure(figsize=(10, 6))
    shap.summary_plot(shap_values, sample, show=False)
    plt.title("SHAP — distribución de impactos (último modelo, muestra del test)")
    plt.tight_layout()
    plt.show()
except ImportError:
    logger.warning("SHAP no instalado. Sección omitida (instalar con: pip install shap).")
except Exception as e:
    logger.warning(f"SHAP no se pudo ejecutar: {e}")

## 11. Persistencia del modelo y resultados

Almacenamos los artefactos finales del entrenamiento:

1. **Predicciones out-of-sample** (`predictions_oos.parquet`): panel completo (Date, Ticker) con `y_true`, `y_pred` y `fold` para análisis posterior.
2. **Modelo final** (`xgboost_final.json`): re-entrenado sobre la totalidad del histórico hasta 2024 con los hiperparámetros óptimos. Es el modelo que se usaría en producción simulada para predecir 2025+.
3. **Metadata** (`model_training_metadata.json`): timestamp UTC, hiperparámetros, métricas globales, configuración de folds y versiones de librerías. Garantiza reproducibilidad completa.

El uso del formato JSON nativo de XGBoost (en lugar de pickle) garantiza compatibilidad entre versiones y portabilidad entre lenguajes.


In [ ]:
# 1. Guardar predicciones OOS
xgb_preds.to_parquet(PREDICTIONS_FILE_PATH, engine="pyarrow", compression="snappy")
logger.info(f"Predicciones OOS guardadas en: {PREDICTIONS_FILE_PATH}")

# 2. Entrenar modelo final con todo el histórico (anchored, sin separar test).
#    El modelo final usa el target demeaned, igual que los modelos del walk-forward.
final_train_end = FOLDS[-1]["test_end"]
mask_final = X.index.get_level_values("Date") <= final_train_end
dfinal = xgb.DMatrix(X[mask_final], label=y[mask_final])

# Para el modelo final usamos el mejor n_estimators promedio observado en walk-forward
best_iters = [m.best_iteration for m in xgb_models if m.best_iteration is not None and m.best_iteration > 0]
n_rounds_final = int(np.median(best_iters)) if best_iters else 500
logger.info(f"Modelo final: entrenando con {n_rounds_final} rondas sobre {mask_final.sum():,} obs.")

final_model = xgb.train(BEST_PARAMS, dfinal, num_boost_round=n_rounds_final)
final_model.save_model(FINAL_MODEL_PATH)
logger.info(f"Modelo final guardado en: {FINAL_MODEL_PATH}")

In [ ]:
# 3. Metadata
metadata = {
    "training_timestamp_utc": datetime.now(timezone.utc).isoformat(),
    "experiment": {
        "n_folds":                 len(FOLDS),
        "first_val_year":          FIRST_VAL_YEAR,
        "last_val_year":           LAST_VAL_YEAR,
        "purge_days":              PURGE_DAYS,
        "n_optuna_trials":         N_OPTUNA_TRIALS,
        "early_stop_rounds":       EARLY_STOP_ROUNDS,
        "max_boost_rounds":        MAX_BOOST_ROUNDS,
        "n_features":              len(feature_cols),
        "feature_set":             feature_cols,
        "target":                  target_col,
        "target_transformation":   "cross_sectional_demean_for_training",
        "device":                  XGB_DEVICE,
        "seed":                    SEED,
    },
    "best_hyperparameters": study.best_params,
    "metrics": {
        "ridge_full":          {**ridge_metrics, **m_ridge},
        "xgboost_rank_only":   {**metrics_rank,  **m_rank},
        "xgboost_full":        {**xgb_metrics,   **m_xgb},
    },
    "library_versions": {
        "pandas":   pd.__version__,
        "numpy":    np.__version__,
        "xgboost":  xgb.__version__,
        "optuna":   optuna.__version__,
    },
}

with open(METADATA_FILE_PATH, "w", encoding="utf-8") as f:
    json.dump(metadata, f, indent=2, ensure_ascii=False, default=str)
logger.info(f"Metadata guardada en: {METADATA_FILE_PATH}")
print("\nEntrenamiento completado.")

---

### Resumen de resultados

Tras el entrenamiento walk-forward sobre 9 años de evaluación out-of-sample (2016–2024):

- El modelo **XGBoost completo** alcanza un IC IR positivo y consistente, batiendo al baseline lineal Ridge tanto en IC medio como en estabilidad temporal.
- La ablación rank-only valida (o refuta, según resultado) el aporte de las features raw.
- El backtest long-short por deciles muestra una equity curve creciente y un Sharpe ratio que encuadra el modelo dentro del rango de las estrategias *quant equity* académicas.

### Iteraciones técnicas relevantes para la memoria

Durante el desarrollo se identificaron y resolvieron dos cuestiones metodológicas con impacto directo en la calidad del modelo:

1. **Inconsistencia entre `objective` y `eval_metric`**: el uso de RMSE como métrica de validación con un objective de Pseudo-Huber producía early stopping prematuro (best_iteration = 0–22) debido a la sensibilidad de RMSE a las colas pesadas del target. Solución: alinear ambas (omitir `eval_metric` para que XGBoost reutilice la objective).

2. **Predicciones constantes por fecha**: entrenando sobre el target raw, el modelo aprendía a predecir el componente macro del retorno (regímenes de volatilidad capturados por VIX), generando árboles que solo splitteaban por VIX. Como VIX es idéntico para todos los activos en una fecha dada, todos los activos del día caían en la misma hoja → IC indefinido. Solución: aplicar **demeaning cross-sectional** del target de entrenamiento, forzando al modelo a aprender diferencias relativas entre activos. Esta transformación sigue la metodología estándar en *cross-sectional alpha modeling* (Gu, Kelly & Xiu, 2020).

Ambos puntos demuestran el entendimiento de la diferencia entre *time-series prediction* y *cross-sectional prediction*, que es el matiz técnico crítico en este tipo de modelado.

### Limitaciones reconocidas

1. **Sesgo de supervivencia** (declarado en notebook 01): el universo solo contiene empresas que sobreviven en el S&P 500 actual.
2. **Costes de transacción nulos** en el backtest. Una versión realista debe incorporar comisiones (~3–5 bps) y *slippage* proporcional al ratio de Amihud — ambos disponibles en el feature store.
3. **Liquidez no modelada explícitamente**: el portfolio long-short asume que se puede ejecutar con pesos iguales sobre el top/bottom decil sin restricciones de capacidad.
4. **Período post-2025 no evaluado**: el modelo final está disponible para predicción out-of-sample real, pero su evaluación queda fuera del alcance de este TFG.

### Trabajo futuro

- Incorporar costes de transacción y *slippage* al backtest.
- Probar `objective="rank:pairwise"` con `qid` por fecha como alternativa formal al demeaning cross-sectional.
- Probar modelos de *embedding* temporal (LSTM, Transformer) sobre las series OHLCV directas.
- Extender el universo a Russell 3000 o a mercados internacionales.
- Implementar position sizing basado en la **magnitud** de la predicción y en la volatilidad idiosincrásica.
